In [11]:
import os
import numpy as np
from lapy import TetMesh
from nibabel.loadsave import load
from neuromodes import EigenSolver
from lapy.plot import plot_tet_mesh
from iso2mesh import vol2mesh
from neuromodes.io import fetch_vol

os.environ["ISO2MESH_TEMP"] = "C:\\Users\\ipop0003\\neuromodes\\neuromodes\\.iso2mesh_temp"
os.makedirs(os.environ["ISO2MESH_TEMP"], exist_ok=True)

In [5]:
def nifti_to_lapy_tetmesh(nifti_path, maxvol=1.0):
    img = load(nifti_path)
    vol = img.get_fdata()

    mask = (vol > 0).astype(np.uint8)

    nx, ny, nz = mask.shape
    ix = np.arange(nx)
    iy = np.arange(ny)
    iz = np.arange(nz)

    node, elem, face = vol2mesh(
        mask,
        ix, iy, iz,
        opt={},
        maxvol=maxvol,
        dofix=1,
        method="cgalsurf"
    )

    affine = img.affine
    spacing = np.sqrt((affine[:3, :3] ** 2).sum(axis=0)) # ASSUMES NO SHEAR; DIAGONAL affine[:3, :3]
    node *= spacing

    tets = elem[:, :4] - 1

    return TetMesh(
        v=node.astype(np.float64),
        t=tets.astype(np.int32)
    )

In [6]:
img = load('C:/Users/ipop0003/neuromodes/neuromodes/data/hcp_hippo-lh_thr25.nii.gz')
print(img.affine)

tetmesh = nifti_to_lapy_tetmesh('C:/Users/ipop0003/neuromodes/neuromodes/data/hcp_hippo-lh_thr25.nii.gz', maxvol=0.5)


[[  -2.    0.    0.   90.]
 [   0.    2.    0. -126.]
 [   0.    0.    2.  -72.]
 [   0.    0.    0.    1.]]
Extracting surfaces from a volume...
Region 1 centroid: [54.5        44.66666667 39.33333333]
Processing threshold level 1...
Iso2mesh meshing utilities do not exist locally, downloading now ...
Extracting C:\Users\ipop0003\iso2mesh-tools\iso2mesh-1.9.8/bin/
Extracting C:\Users\ipop0003\iso2mesh-tools\iso2mesh-1.9.8/bin/PoissonRecon.mexa64
Setting permission C:\Users\ipop0003\iso2mesh-tools\iso2mesh-1.9.8/bin/PoissonRecon.mexa64
Extracting C:\Users\ipop0003\iso2mesh-tools\iso2mesh-1.9.8/bin/PoissonRecon.mexmaci64
Setting permission C:\Users\ipop0003\iso2mesh-tools\iso2mesh-1.9.8/bin/PoissonRecon.mexmaci64
Extracting C:\Users\ipop0003\iso2mesh-tools\iso2mesh-1.9.8/bin/PoissonRecon_x86-64.exe
Setting permission C:\Users\ipop0003\iso2mesh-tools\iso2mesh-1.9.8/bin/PoissonRecon_x86-64.exe
Extracting C:\Users\ipop0003\iso2mesh-tools\iso2mesh-1.9.8/bin/README.txt
Setting permission C:\

In [18]:
# Compute modes and plot
volser = EigenSolver(tetmesh).solve(20, seed=0)

plot_tet_mesh(tetmesh, vfunc=-volser.emodes[:,10], plot_edges=True)

Found 2684 triangles on boundary.
Flipped 6378 tetrahedra
Found 2684 triangles on boundary.


In [19]:
# Compare to hippocampal volume mesh from BrainEigenmodes repo
hippo_vol = fetch_vol("hippocampus")
volser2 = EigenSolver(hippo_vol).solve(20, seed=0)
plot_tet_mesh(hippo_vol, vfunc=volser2.emodes[:,9], plot_edges=True)

--> VTK format         ... 
 --> DONE ( V: 1449 , T: 5168 )

Found 2056 triangles on boundary.
Mesh is oriented, nothing to do
Found 2056 triangles on boundary.
